In [3]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f'{train_df.describe()}\n')
print(f'{train_df.info()}\n')
print(f'{train_df.isnull().sum()}\n')

                id      gas_raw       temp_c        flame      gas_avg  \
count  3232.000000  2325.000000  2322.000000  3232.000000  2340.000000   
mean   7016.380260   599.895891    70.574290     0.249691   581.776136   
std    1151.040797   406.219182    52.912426     0.432901   385.839546   
min    5000.000000  -155.204229   -21.952217     0.000000  -124.847898   
25%    6009.750000   275.473835    26.910714     0.000000   271.356831   
50%    7026.500000   600.083115    73.652402     0.000000   454.164968   
75%    8013.250000   855.381985   104.833014     0.000000   840.667084   
max    8999.000000  2172.368187   268.381581     1.000000  2010.299948   

          temp_avg         dGas        dTemp  sensor_noise  timestamp_ms  \
count  2327.000000  2263.000000  2267.000000   2324.000000  3.232000e+03   
mean     69.687802     0.822007     0.024979      0.212370  1.498010e+09   
std      52.393391    27.252869     1.595192      1.318669  2.844246e+08   
min     -19.473057  -126.9559

In [ ]:
hazards = ['NORMAL', 'GAS_LEAK', 'OVERHEAT', 'FIRE']

citiri = []
for h in hazards:
    n = (train_df['label'] == h).sum()
    citiri.append(n/len(train_df['label']))
med_temps = train_df.groupby('label')['temp_c'].mean().round(2).tolist()
med_gas = train_df.groupby('label')['gas_raw'].mean().round(1).tolist()

T1 = pd.DataFrame({
    'subtaskID':1,
    'datapointID':[1,2,3,4,5,6,7,8,9,10,11,12],
    'answer': citiri+med_temps+med_gas
})

In [ ]:
na_cols = ['gas_raw','temp_c','gas_avg','temp_avg','dGas','dTemp','sensor_noise',
           'battery_level','vehicle_speed','gas_sensor_drift','ambient_temp']

for nc in na_cols:
    train_df[nc] = train_df[nc].fillna(train_df[nc].median())

In [18]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

train_df.drop(columns=(['id','sensor_id']))

X = train_df.drop(columns=('label'))
y = train_df['label']

num_cols = X.select_dtypes(include=['float64','int64']).columns.tolist()

preprocessor = ColumnTransformer(transformers=(
    ('numerical',StandardScaler(),num_cols),
    ('text',OneHotEncoder(),['weather_condition'])
))

X_processed = preprocessor.fit_transform(X)

label = LabelEncoder()
y = label.fit_transform(y)

In [23]:
X_processed_df = pd.DataFrame(X_processed,columns=preprocessor.get_feature_names_out())
y_series = pd.Series(y).reset_index(drop=True)

correlations = X_processed_df.corrwith(y_series, method='spearman').abs()

df_corr = pd.DataFrame({
    'feature':correlations.index,
    'corr_val':correlations.values
})

df_corr = df_corr.sort_values(
    by=['corr_val','feature'],
    ascending=[False,True]
)

top6 = df_corr['feature'].head(6).tolist()

In [ ]:
cod = label.transform(['NORMAL'])[0]
normal = X_processed_df[y == cod]

normal = normal[top6]

from scipy.spatial.distance import mahalanobis

